# Monte Carlo policy evaluation: lab notebook

**Machine Learning (5CS4-05), Semi-supervised and reinforcement learning**

Run the cells from top to bottom (Runtime > Run all). NumPy and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import numpy as np

### 1. The classic 3 x 4 grid world (reward is received on ENTERING a cell)

In [ ]:
# 1. The classic 3 x 4 grid world (reward is received on ENTERING a cell)
rows = ["...G", ".#.X", "S..."]
R, C = len(rows), len(rows[0])
STEP, SLIP = -0.04, 0.2
ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]          # up, right, down, left
ARROW = "^>v<"
cell = lambda s: rows[s // C][s % C]
terminal = lambda s: cell(s) in "GX"
wall = lambda s: cell(s) == "#"
reward = lambda s: {"G": 1.0, "X": -1.0}.get(cell(s), STEP)

def move(s, a):
    r, c = s // C + ACTIONS[a][0], s % C + ACTIONS[a][1]
    return s if not (0 <= r < R and 0 <= c < C) or rows[r][c] == "#" else r * C + c

def transitions(s, a):
    """list of (probability, next_state, reward)"""
    if terminal(s) or wall(s):
        return [(1.0, s, 0.0)]
    out = [(1 - SLIP, a), (SLIP / 2, (a + 1) % 4), (SLIP / 2, (a + 3) % 4)]
    return [(p, move(s, b), reward(move(s, b))) for p, b in out]

def q_value(s, a, V, gamma):
    return sum(p * (r + gamma * V[s2]) for p, s2, r in transitions(s, a))

def show(V):
    for i in range(R):
        print("  ".join(f"{V[i * C + j]:6.3f}" if not wall(i * C + j) else "  wall" for j in range(C)))

nS = R * C
gamma = 0.9
rng = np.random.default_rng(0)

### 2. A policy to evaluate: the optimal policy found by value iteration

In [ ]:
# 2. A policy to evaluate: the optimal policy found by value iteration
V = np.zeros(nS)
for _ in range(1000):
    V = np.array([0 if terminal(s) or wall(s) else max(q_value(s, a, V, gamma) for a in range(4)) for s in range(nS)])
pi = {s: int(np.argmax([q_value(s, a, V, gamma) for a in range(4)])) for s in range(nS) if not terminal(s) and not wall(s)}

### 3. Generate an episode by sampling the environment (we only use samples, not the model)

In [ ]:
# 3. Generate an episode by sampling the environment (we only use samples, not the model)
def step(s, a):
    p, s2, r = zip(*transitions(s, a))
    k = rng.choice(len(p), p=p)
    return s2[k], r[k]

def episode(start):
    s, states, rewards = start, [], []
    while not terminal(s):
        s2, r = step(s, pi[s]); states.append(s); rewards.append(r); s = s2
    return states, rewards

### 4. First-visit and every-visit Monte Carlo

In [ ]:
# 4. First-visit and every-visit Monte Carlo
def mc(n_episodes, first_visit=True):
    total, count = np.zeros(nS), np.zeros(nS)
    starts = list(pi.keys())
    for _ in range(n_episodes):
        states, rewards = episode(starts[rng.integers(len(starts))])
        G, returns = 0.0, [0.0] * len(states)
        for t in reversed(range(len(states))):          # G_t = r_{t+1} + gamma * G_{t+1}
            G = rewards[t] + gamma * G; returns[t] = G
        seen = set()
        for t, s in enumerate(states):
            if first_visit and s in seen:
                continue
            seen.add(s); total[s] += returns[t]; count[s] += 1
    return total / np.maximum(count, 1)

for n in (10, 100, 1000, 10000):
    est = mc(n)
    err = np.sqrt(np.mean([(est[s] - V[s]) ** 2 for s in pi]))
    print(f"{n:6d} episodes, first-visit: RMS error = {err:.4f}")
print("every-visit, 10000 episodes: RMS error =", round(float(np.sqrt(np.mean([(mc(10000, False)[s] - V[s]) ** 2 for s in pi]))), 4))
print("estimate at start (2,0):", round(float(mc(10000)[2 * C]), 3), " exact:", round(float(V[2 * C]), 3))

### 5. A returns example worked by hand

In [ ]:
# 5. A returns example worked by hand
rewards = [0, 0, 10]
G = 0
for r in reversed(rewards):
    G = r + 0.5 * G
print("return of [0, 0, 10] with gamma = 0.5:", G)

## Expected output

Running all cells above should print:

```
    10 episodes, first-visit: RMS error = 0.1346
   100 episodes, first-visit: RMS error = 0.0756
  1000 episodes, first-visit: RMS error = 0.0120
 10000 episodes, first-visit: RMS error = 0.0056
every-visit, 10000 episodes: RMS error = 0.0067
estimate at start (2,0): 0.388  exact: 0.374
return of [0, 0, 10] with gamma = 0.5: 2.5
```

## Practice exercises

Try these in the cells below.

1. Compare first-visit and every-visit estimates after 50 episodes. Which is closer to the DP value?
2. Plot the RMS error against the number of episodes (log axis). What is the slope?
3. Change gamma to 1 and see why the episodes must terminate.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
